# 04 · Knowledge distillation: a big teacher trains a small student

**Goals of this notebook**

1. Understand the teacher-student idea and what "dark knowledge" is.
2. Train a powerful **teacher**: a convolutional network that is too expensive for our challenge.
3. Learn the **softmax temperature** and the **distillation loss**.
4. Distil the teacher into small MLP **students**, and find out *when* it helps: with plenty of data, or with little data?

In [ ]:
import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# EXTERNAL (shared with the other notebooks, code shown in notebook 01):
#   load_gesture_data, limit_training_data  the synthetic sensor data and a helper that
#                                           keeps only n examples per gesture
#                                           (see the comment block in notebook 01)
#   MLP, count_params, count_macs           the small network class and its counters
#   accuracy, predict_logits, set_seed      test accuracy, raw network outputs, repeatable randomness
# Why external: used in every notebook and explained line by line in notebook 01.
# Everything about distillation (the ConvNet teacher, the loss and the teacher-aware
# training loop) is written out below, because that is the topic of THIS notebook.
# ---------------------------------------------------------------------------
import sys, pathlib
# Only needed so Python can find the course folder `aiedge` (the helpers imported below).
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from aiedge.data import load_gesture_data, limit_training_data
from aiedge.models import MLP, count_params, count_macs
from aiedge.training import accuracy, set_seed, predict_logits

GESTURES = ("punch", "lift", "wave", "circle", "twist")   # the 5 classes
WINDOW_SAMPLES, N_CHANNELS = 50, 6                         # 50 readings of 6 sensor channels
MCU_CLOCK_HZ = 64_000_000                                  # Arduino Nano 33 BLE Sense
CYCLES_PER_MAC = 4                                         # rough cost of one multiply-accumulate

data = load_gesture_data()

## 1. The idea

A small network trained on the labels alone gets very little information per example: "this is a *punch*". Full stop.

A big, well-trained network knows more. For a sloppy punch it might say: *"80% punch, 15% lift, 5% circle; it went a bit upwards too."* These **soft targets** tell the student *how similar* the gestures are and which examples are borderline cases. Geoffrey Hinton called this information **dark knowledge** (Hinton, Vinyals & Dean, *Distilling the Knowledge in a Neural Network*, 2015).

**Knowledge distillation (KD)**:
1. Train a big, accurate **teacher**. Its cost doesn't matter: it never goes to the device.
2. Train the small **student** (which does go to the device) to imitate the teacher's soft outputs, and also to get the true labels right.

The teacher can even have a completely **different architecture** than the student. We will use a convolutional network as teacher, and an MLP as student.

## 2. The teacher: a 1-D convolutional network

An MLP treats its 300 inputs as 300 unrelated numbers. It has no idea that reading 10 and reading 11 are neighbours in time, or that "a spike in `ax`" means the same thing whether it happens at 0.2 s or at 0.3 s. It has to learn this separately for every position.

A **convolutional layer** instead slides a small **filter** (here 5 readings long) along the time axis. It is a little pattern detector, for example "sharp rise in `ax`", and it uses **the same weights at every position**. **Pooling** then summarises: "was this pattern present anywhere?" This makes convolutional networks naturally good at signals and images.

The price: the filters are applied at every time step, so the network needs many more MACs, and more RAM for the intermediate results.

### The teacher in code

Here is the convolutional teacher. Read it top to bottom: three `Conv1d` layers (each followed by `ReLU`, the first two also by pooling) find patterns in time; then we summarise every filter over the whole window (average and maximum) and a final `Linear` layer turns that summary into 5 scores, one per gesture. `count_macs_conv` counts the multiply-accumulate operations, as in notebook 01 but for convolutions: every filter position costs `in_channels × kernel_size` MACs.

In [ ]:
class ConvNet(nn.Module):
    """1-D convolutional network, used only as a *teacher* (it never goes to the board)."""
    def __init__(self, channels=(32, 64), kernel_size=5, dropout=0.3, n_classes=5):
        super().__init__()
        c1, c2 = channels
        pad = kernel_size // 2                       # keeps the length unchanged
        self.features = nn.Sequential(
            nn.Conv1d(N_CHANNELS, c1, kernel_size, padding=pad), nn.ReLU(),
            nn.MaxPool1d(2),                          # 50 -> 25 time steps
            nn.Conv1d(c1, c2, kernel_size, padding=pad), nn.ReLU(),
            nn.MaxPool1d(2),                          # 25 -> 12 time steps
            nn.Conv1d(c2, c2, kernel_size, padding=pad), nn.ReLU(),
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(2 * c2, n_classes)

    def forward(self, x):
        # (N, 300) -> (N, 50 time steps, 6 channels) -> (N, 6, 50), the layout Conv1d wants
        x = x.view(-1, WINDOW_SAMPLES, N_CHANNELS).transpose(1, 2)
        x = self.features(x)
        x = torch.cat([x.mean(dim=2), x.amax(dim=2)], dim=1)   # summarise over time
        return self.head(self.dropout(x))


def count_macs_conv(model):
    """Multiply-accumulates of the ConvNet for one prediction."""
    macs, length = 0, WINDOW_SAMPLES
    for layer in model.features:
        if isinstance(layer, nn.Conv1d):
            macs += layer.in_channels * layer.out_channels * layer.kernel_size[0] * length
        elif isinstance(layer, nn.MaxPool1d):
            length //= 2
    return macs + model.head.in_features * model.head.out_features

### The training loop, with an optional teacher

This is the training loop of notebook 01 (mini-batches, Adam, early stopping on the validation set), with one extra option: **`teacher`**. If a teacher is given, the loss is the *distillation loss* (defined in section 4) instead of plain cross-entropy, and the teacher looks at **exactly the same batch** as the student. The `random_time_shift` function is the data augmentation from notebook 01: it moves every gesture a few readings earlier or later.

In [ ]:
def random_time_shift(x, max_shift=6):
    """Data augmentation: move every gesture in the batch a few readings earlier/later."""
    b = x.shape[0]
    w = x.view(b, WINDOW_SAMPLES, N_CHANNELS)
    shifts = torch.randint(-max_shift, max_shift + 1, (b,))
    t = torch.arange(WINDOW_SAMPLES)
    src = (t[None, :] - shifts[:, None]).clamp(0, WINDOW_SAMPLES - 1)   # repeat the edge values
    return torch.gather(w, 1, src[:, :, None].expand(-1, -1, N_CHANNELS)).reshape(b, -1)


def train(model, data, epochs=150, lr=3e-3, batch_size=64, weight_decay=1e-4, patience=25,
          teacher=None, temperature=4.0, alpha=0.5, augment=False, seed=0, verbose=True):
    """Train `model`; with `teacher=...` it learns from the teacher's soft outputs as well."""
    if seed is not None:
        set_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if teacher is not None:
        teacher.eval()                                   # the teacher is never trained here
    best_acc, best_state, bad_epochs = -1.0, None, 0
    n = len(data.X_train)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(n)                        # new random order every epoch
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            xb, yb = data.X_train[idx], data.y_train[idx]
            if augment:
                xb = random_time_shift(xb)
            logits = model(xb)
            if teacher is None:
                loss = F.cross_entropy(logits, yb)       # labels only
            else:
                with torch.no_grad():
                    teacher_logits = teacher(xb)         # the teacher sees the same (shifted) batch
                loss = distillation_loss(logits, teacher_logits, yb, temperature, alpha)
            opt.zero_grad()
            loss.backward()
            opt.step()

        val_acc = accuracy(model, data.X_val, data.y_val)
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(f"epoch {epoch:3d}  val acc {val_acc:.3f}")
        if val_acc > best_acc:                           # remember the best epoch
            best_acc, best_state, bad_epochs = val_acc, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:                   # early stopping
                break
    model.load_state_dict(best_state)                    # go back to the best epoch
    return best_acc

In [ ]:
set_seed(0)
teacher = ConvNet()
print(teacher)
hist = train(teacher, data, augment=True, verbose=False)
teacher_acc = accuracy(teacher, data.X_test, data.y_test)

teacher_macs = count_macs_conv(teacher)
print(f"\nteacher test accuracy: {teacher_acc:.1%}")
print(f"teacher: {count_params(teacher):,} parameters, {teacher_macs:,} MACs "
      f"-> est. {teacher_macs * CYCLES_PER_MAC / MCU_CLOCK_HZ * 1e3:.0f} ms per prediction on the board")

set_seed(0)
baseline = MLP(hidden=(64, 32))
train(baseline, data, augment=True, verbose=False)
base_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline MLP: {base_acc:.1%}, {count_params(baseline):,} parameters, {count_macs(baseline):,} MACs")

The teacher is clearly more accurate than our baseline MLP, but it needs ~25x more computation per prediction. In this course, it is the expert we learn from.

## 3. Soft targets and temperature

Let's look at what the teacher says about a few test examples:

In [ ]:
with torch.no_grad():
    t_logits = predict_logits(teacher, data.X_test)
t_probs = torch.softmax(t_logits, dim=1)
confidence = t_probs.max(dim=1).values
order = confidence.argsort()          # least confident first
examples = [int(order[-1]), int(order[len(order) // 20]), int(order[0])]   # most sure, 5th percentile, least sure

print(f"{'':28s}" + "".join(f"{g:>8s}" for g in GESTURES))
for i in examples:
    label = f"true: {GESTURES[data.y_test[i]]}"
    print(f"{label:28s}" + "".join(f"{p:8.3f}" for p in t_probs[i].numpy()))

For most examples (first row) the teacher is practically certain. Then the soft target says nearly the same as the hard label. For the harder ones (other rows) the teacher spreads its probability over the gestures that look alike. To bring out the information hidden in the small probabilities, distillation **softens** the softmax with a **temperature** $T$:

$$p_k(T) = \frac{e^{z_k / T}}{\sum_j e^{z_j / T}}$$

$T = 1$ is the normal softmax. A larger $T$ makes the distribution flatter, so the differences between the *unlikely* gestures become visible: "lift is much more likely than wave".

In [ ]:
i = examples[1]
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, T in zip(axes, [1, 2, 4, 8]):
    p = torch.softmax(t_logits[i] / T, dim=0).numpy()
    ax.bar(GESTURES, p, color="C0")
    ax.set_title(f"T = {T}"); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("probability")
fig.suptitle(f"Teacher's soft targets for one '{GESTURES[data.y_test[i]]}' example", y=1.05)
plt.show()

## 4. The distillation loss

The student is trained on a mix of two losses:

$$\mathcal{L} = \alpha \cdot \underbrace{\text{CE}\big(y,\ p^{s}\big)}_{\text{hard loss: match the labels}} \;+\; (1 - \alpha)\, T^2 \cdot \underbrace{\text{KL}\big(p^{t}(T)\ \|\ p^{s}(T)\big)}_{\text{soft loss: match the teacher}}$$

* $\text{CE}$ is the usual cross-entropy with the true label (notebook 01).
* $\text{KL}$, the *Kullback-Leibler divergence*, measures how different two probability distributions are (0 if identical). Here: the student's softened output $p^s(T)$ vs the teacher's $p^t(T)$.
* $\alpha$ balances the two. We use $\alpha = 0.5$: listen equally to the labels and to the teacher.
* Why $T^2$? Softening by $T$ makes the gradients of the soft loss about $T^2$ times smaller. Multiplying by $T^2$ keeps both parts of the loss on the same scale when you change $T$.

Here is the implementation. It is short:

In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.5):
    # hard loss: ordinary cross-entropy with the true labels
    hard = F.cross_entropy(student_logits, labels)
    # soft loss: KL divergence between the softened teacher and student distributions
    # (kl_div wants the student as log-probabilities and the teacher as probabilities)
    soft = F.kl_div(F.log_softmax(student_logits / temperature, dim=1),
                    F.softmax(teacher_logits / temperature, dim=1),
                    reduction="batchmean") * temperature**2
    return alpha * hard + (1 - alpha) * soft

**Temperature in the code:** `student_logits / temperature` divides the raw scores before the softmax. `temperature=1` changes nothing; `4` flattens the probabilities (as in the plot above), which is what makes the small ones visible to the student.

During training, the `train(student, data, teacher=teacher)` loop above lets the teacher look at **the same augmented batch** as the student. That matters: for every randomly shifted example, the student gets the teacher's opinion about *that exact* input. The teacher's robustness to time shifts is transferred to the student this way.

In [ ]:
# A tiny check of the formula: a student that copies the teacher exactly has soft loss 0.
labels = data.y_test[:8]
same = distillation_loss(t_logits[:8], t_logits[:8], labels, temperature=4.0, alpha=0.0)
print(f"soft loss when student == teacher: {abs(same.item()):.6f}")

## 5. Experiment 1: distillation with plenty of data

Let's distil the teacher into MLP students of several sizes and compare each with the same student trained without a teacher, using exactly the same settings (including augmentation). One run can be lucky or unlucky (remember the ±1 point rule from notebook 01), so every result is the **mean of three seeds**, ± the spread (standard deviation). This cell takes a minute or two.

In [ ]:
def compare_students(train_data, teacher, sizes, seeds=(0, 1, 2)):
    """Mean and spread of test accuracy, without and with the teacher, for every student size."""
    results = {"no teacher": [], "distilled": []}
    for h in sizes:
        accs = {"no teacher": [], "distilled": []}
        for s in seeds:
            set_seed(s); m = MLP(hidden=h)
            train(m, train_data, augment=True, seed=s, verbose=False)
            accs["no teacher"].append(accuracy(m, data.X_test, data.y_test))
            set_seed(s); m = MLP(hidden=h)
            train(m, train_data, teacher=teacher, temperature=4.0, alpha=0.5, augment=True, seed=s, verbose=False)
            accs["distilled"].append(accuracy(m, data.X_test, data.y_test))
        for k in results:
            results[k].append(float(np.mean(accs[k])))
        print(f"student {str(h):9s} ({count_params(m):6,} params): "
              f"no teacher {np.mean(accs['no teacher']):.1%} ± {np.std(accs['no teacher']):.1%}   "
              f"distilled {np.mean(accs['distilled']):.1%} ± {np.std(accs['distilled']):.1%}")
    return results

student_sizes = [(64, 32), (32, 16), (16, 8), (8,)]
student_params = [count_params(MLP(hidden=h)) for h in student_sizes]
full = compare_students(data, teacher, student_sizes)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
for (name, accs), style in zip(full.items(), ["o-", "s--"]):
    ax.plot(student_params, accs, style, label=name)
ax.axhline(teacher_acc, color="gray", ls=":", lw=1.5)
ax.text(student_params[-1], teacher_acc, "teacher", va="bottom", color="gray")
ax.set_xscale("log"); ax.set_xlabel("student parameters (log scale)")
ax.set_ylabel("test accuracy (mean of 3 seeds)")
ax.set_title("Distillation with 560 examples per gesture"); ax.legend(); plt.show()

**Hmm.** With our full training set (560 examples per gesture), distillation makes **no consistent difference**. Some students gain a little, others lose a little, and the differences (up to about a point) are similar to the spread between seeds. Note one thing distillation often *does* do: the distilled students vary less between seeds. Why no clear gain?

With this much data, even a small MLP learns almost everything there is to learn: the students are already within one or two points of the teacher. There is little "dark knowledge" left to transfer.

That's an important, honest result: **distillation is not magic.** Papers often show big gains, typically for hard tasks with a big gap between teacher and student. Whether it helps *your* problem, you have to measure, with several seeds.

## 6. Experiment 2: distillation with little data

Now a situation you will actually face: you recorded your own gestures, and you have only a few dozen examples per gesture (notebook 01, section 8). With so few examples, a network easily *overfits*: it memorises the training examples instead of learning what a gesture really looks like.

We simulate this by keeping only **20, 40 or 80 training examples per gesture** (`limit_training_data`). The validation and test sets stay the same, so the results are comparable. For every amount we train a new teacher on the *same* small training set (the teacher gets no extra information!), and then a 32-16 student with and without that teacher, three seeds each. This takes about a minute.

In [ ]:
amounts = [20, 40, 80]
small = {"no teacher": [], "distilled": []}
small_teacher_accs = []
for n in amounts:
    small_data = limit_training_data(data, n_per_class=n)
    set_seed(0)
    small_teacher = ConvNet()
    train(small_teacher, small_data, augment=True, verbose=False)
    small_teacher_accs.append(accuracy(small_teacher, data.X_test, data.y_test))
    print(f"--- {n} examples per gesture ({len(small_data.y_train)} in total), "
          f"teacher {small_teacher_accs[-1]:.1%}")
    r = compare_students(small_data, small_teacher, [(32, 16)])
    for k in small:
        small[k] += r[k]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.8))
for (name, accs), style in zip({"no teacher": small["no teacher"], "distilled": small["distilled"],
                                "teacher": small_teacher_accs}.items(), ["o-", "s--", "^:"]):
    ax.plot(amounts, accs, style, label=name)
ax.set_xlabel("training examples per gesture"); ax.set_ylabel("test accuracy (mean of 3 seeds)")
ax.set_title("Student 32-16 with little data"); ax.set_xticks(amounts); ax.legend(); plt.show()
for n, a, b in zip(amounts, small["no teacher"], small["distilled"]):
    print(f"{n:3d} examples per gesture: distillation gain {100 * (b - a):+.1f} points")

**Now distillation clearly helps, and the less data, the more it helps.** Typical gains are a few points with 20 examples per gesture, about one point with 40, and nothing left by about 80 (as in experiment 1). Your exact numbers will vary a little.

### Why does it help here, and not in experiment 1?
* **Built-in knowledge.** The convolutional teacher has a built-in assumption ("a pattern means the same thing wherever it occurs in time"). It generalises better from few examples than an MLP, so the gap between teacher and student is bigger, and there is more to transfer.
* **More information per example**: a full probability distribution ("80% punch, 15% lift") instead of one label.
* **The teacher labels new inputs.** Thanks to augmentation, the student sees a randomly *shifted* version of each example in every epoch, and the teacher gives its opinion about *that exact* input. In effect the teacher turns 40 recordings into a never-ending stream of labelled training examples.

### And the capacity gap
For the *smallest* students, distillation can even hurt: a student that is too small cannot imitate the teacher's complex function, and trying to do so distracts it from the labels. This is known as the **capacity gap**. Common remedies are a "teaching assistant" (distil first into a medium model, then from that one into the tiny one), or a larger $\alpha$ so the labels count more.

**Conclusion**: distillation is most valuable when **labelled data is scarce** and the teacher has an advantage the student lacks. In notebook 05 we always use it: it costs nothing on the device, and it matters most when you train on your own (small) recordings.

In [ ]:
(ROOT / "models").mkdir(exist_ok=True)
torch.save(teacher.state_dict(), ROOT / "models" / "teacher.pt")
print("teacher saved to models/teacher.pt")

## Summary

* KD trains a small **student** to imitate a big **teacher**'s soft outputs as well as the labels.
* The **temperature** $T$ softens the probabilities and reveals the "dark knowledge"; the loss is $\alpha\,\text{CE} + (1-\alpha)T^2\,\text{KL}$.
* The teacher can be any architecture, as expensive as you like, since it never leaves your laptop.
* With plenty of data the gain here is negligible; with little labelled data (like your own recordings) it is up to a few points, at zero cost on the device.
* Beware the **capacity gap** for very small students, and always compare over several seeds.

## Exercises

1. ⭐ **Temperature.** Using `limit_training_data(data, 20)` and a teacher trained on it, distil the 32-16 student with $T = 1, 2, 4, 8$ (keep $\alpha = 0.5$). Which works best? What happens at $T = 1$, and why?
2. ⭐ **α = 1.** What does `alpha=1.0` mean? Predict the result first, then check.
3. **A different teacher.** Use `MLP(hidden=(256, 128), dropout=0.3)` as the teacher instead of the ConvNet. Is it a better or worse teacher? Is the best *model* always the best *teacher*?
4. **A bad teacher.** Train a teacher for only 2 epochs (`epochs=2`) and distil from it. What happens to the student?
5. **No augmentation.** Repeat experiment 2 with `augment=False` for both students. Is the gain from distillation bigger or smaller? Why? (Hint: see "the teacher labels new inputs".)
6. **Student size.** Repeat experiment 2 with a 16-8 and a 64-32 student. Does the gain from distillation depend on the student's size?